In [1]:
# ============================================================
# CS781 Capstone - PHASE 3: Rhetorical Role Prediction (RR)
# One label per sentence, CL + IT domains. Starts from YOUR Phase-1 model (best.pt).
# EVERYTHING is implemented from scratch in plain PyTorch (no peft / bitsandbytes) - Phase 3 rule.
#
# One notebook for all experiments - change Cell 1 only:
#   FT_METHOD: "full" | "lora" | "rslora" | "qlora" | "adalora" | "lorma" | "prompt" | "prefix" | "ia3"
#   Knowledge distillation: set KD_TEACHER to a trained Phase-3 run's best_rr.pt
# ============================================================

# %% 0. IMPORTS
import os, json, math, time, glob, random, zipfile
from collections import defaultdict, Counter
from types import SimpleNamespace
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import PreTrainedTokenizerFast   # tokenizer loading only (your own BPE file)

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = None

# %% 1. CONFIG  <-------------------------------------------------------- EDIT HERE
QUICK_TEST = False          # True = few docs, 1 epoch, to check the pipeline

MODEL_TYPE = "gqa"        # "gpt" (Model 1) | "llama" (Model 2) | "gqa" (Model 3) | "hybrid" (Model 4)
PHASE1_CKPT = "/kaggle/input/datasets/kavyajain123987/phase1-models/best Phase1 model3 gqa swa.pt"
TOKENIZER = "/kaggle/input/datasets/kavyajain123987/phase1/tokenizer (1).json"

FT_METHOD = "lorma"          # full | lora | rslora | qlora | adalora | lorma | prompt | prefix | ia3
LORA_R, LORA_ALPHA = 8, 16  # lora / rslora / qlora / lorma rank + scaling
ADA_R_INIT, ADA_R_TARGET = 12, 4   # adalora: start rank per layer -> average target rank
N_PROMPT = 16               # prompt / prefix: number of virtual tokens
QUANT_BLOCK = 64            # qlora: block size for 4-bit absmax quantization

CONTEXT = "doc"             # "doc" = sentences read in document order (left context) | "sentence" = each alone
POOL = "last"               # sentence vector: "last" token (sentence-end marker) | "mean" of its tokens
HEAD = "bigru"              # "bigru" = BiGRU over sentence vectors (adds right context) | "linear"
CLASS_WEIGHT = "sqrt_inv"   # None | "sqrt_inv" (rare roles weighted up)
DOMAINS = ["CL", "IT"]      # train on both domains (evaluated separately too)

MAX_LEN = 512               # tokens per window (+ N_PROMPT must fit the Phase-1 context)
OVERLAP = 128
MAX_SENT_TOK = 128          # very long sentences are truncated to this many tokens
WIN_BATCH = 8               # windows per forward pass
EPOCHS = 12
LR_BODY = 1e-4              # full fine-tuning / KD student body
LR_NEW = 1e-3               # head + all adapter parameters
WEIGHT_DECAY = 0.01
DOCS_PER_STEP = 2           # gradient accumulation over training segments
TRAIN_SEG_TOK = 3072        # training: long documents are cut into segments of consecutive sentences of at
                            # most this many tokens (whole 70k-token docs don't fit in GPU memory with grads).
                            # Prediction always uses the full document (no gradients -> no memory problem).
SEED = 42

# ---- knowledge distillation (mandatory experiment) ----
KD_TEACHER = None           # path to a trained Phase-3 run's best_rr.pt (file, .zip, or extracted folder)
KD_KEEP_EVERY = 2           # student = Phase-1 model keeping every 2nd block (8 -> 4)
KD_T, KD_ALPHA = 2.0, 0.5   # loss = ALPHA * CE(gold) + (1-ALPHA) * T^2 * KL(teacher || student)

OUTPUT_DIR = f"/kaggle/working/phase3_{MODEL_TYPE}_{FT_METHOD}" + ("_kd" if KD_TEACHER else "")
os.makedirs(OUTPUT_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

# %% 2. DOWNLOAD PHASE-3 DATA + LOOK AT IT
from huggingface_hub import snapshot_download
DATA_DIR = snapshot_download("Exploration-Lab/CS-781-Capstone", repo_type="dataset", token=HF_TOKEN,
                             allow_patterns=["Phase_3_RR/*"], local_dir="/tmp/cs781_data")
P3 = os.path.join(DATA_DIR, "Phase_3_RR")
for root, _, files in os.walk(P3):
    for f in sorted(files):
        print(os.path.join(root, f), f"{os.path.getsize(os.path.join(root, f)) / 1e6:.2f} MB")

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def peek(obj, n=400):
    s = json.dumps(obj, ensure_ascii=False)
    return s[:n] + ("..." if len(s) > n else "")

for name in ["Train/CL_train.json", "Test/CL_test.json"]:
    raw = load_json(os.path.join(P3, name))
    print(f"\n{name}:", type(raw).__name__, len(raw), "| first:",
          peek(raw[0] if isinstance(raw, list) else next(iter(raw.items()))))

# %% 3. PARSE DATA
lv = load_json(os.path.join(P3, "Train/label_vocab.json"))
if isinstance(lv, dict):
    LABELS = [k for k, _ in sorted(lv.items(), key=lambda kv: kv[1])] if all(isinstance(v, int) for v in lv.values()) \
        else list(lv.get("labels", lv.keys()))
else:
    LABELS = list(lv)
LAB2ID = {l: i for i, l in enumerate(LABELS)}
print("Labels:", LABELS)

def parse_docs(obj, domain):
    """-> list of {"id", "domain", "sents", "labels" (or None)}"""
    if isinstance(obj, dict) and "data" in obj and isinstance(obj["data"], (list, dict)):
        obj = obj["data"]
    items = obj.items() if isinstance(obj, dict) else enumerate(obj)
    docs = []
    for k, d in items:
        if isinstance(d, list):                       # {"doc_id": ["sent1", "sent2", ...]}
            d = {"text": d}
        if not isinstance(d, dict):
            continue
        sents = next((d[key] for key in ("text", "sentences", "sents", "document") if key in d), None)
        if isinstance(sents, str):
            sents = [sents]
        if sents is None:
            print("WARNING: no sentence list in doc", k, "keys:", list(d)[:10]); continue
        labs = next((d[key] for key in ("labels", "label", "roles") if key in d), None)
        doc_id = d.get("id", d.get("doc_id", d.get("document_id", k)))
        docs.append({"id": str(doc_id), "domain": domain,
                     "sents": [str(x) for x in sents], "labels": list(labs) if labs is not None else None})
    return docs

splits = {"train": [], "dev": [], "test": []}
for dom in DOMAINS:
    splits["train"] += parse_docs(load_json(os.path.join(P3, f"Train/{dom}_train.json")), dom)
    splits["dev"] += parse_docs(load_json(os.path.join(P3, f"Train/{dom}_dev.json")), dom)
    splits["test"] += parse_docs(load_json(os.path.join(P3, f"Test/{dom}_test.json")), dom)

for name in ("train", "dev"):
    for d in splits[name]:
        assert len(d["sents"]) == len(d["labels"]), f"{d['id']}: sentence/label count mismatch"
        bad = [l for l in d["labels"] if l not in LAB2ID]
        assert not bad, f"unknown labels {set(bad)}"
assert splits["test"], "No test documents parsed - send the 'Test/CL_test.json: ... first:' line printed above"
test_ids = [d["id"] for d in splits["test"]]
if len(set(test_ids)) != len(test_ids):
    print("WARNING: duplicate test doc IDs across CL/IT - predictions would overwrite each other!")
for name, ds in splits.items():
    n_s = sum(len(d["sents"]) for d in ds)
    print(f"{name}: {len(ds)} docs, {n_s} sentences | per domain:", dict(Counter(d["domain"] for d in ds)))
cnt = Counter(l for d in splits["train"] for l in d["labels"])
print("Train label counts:", {l: cnt[l] for l in LABELS})

if QUICK_TEST:
    splits["train"], splits["dev"] = splits["train"][:6], splits["dev"][:3]
    EPOCHS = 1

# %% 4. TOKENIZER + DOCUMENT ENCODING
tokenizer = PreTrainedTokenizerFast(tokenizer_file=TOKENIZER)
tokenizer.model_max_length = int(1e9)
SEP = tokenizer.convert_tokens_to_ids("<|endoftext|>")
if SEP is None or SEP == tokenizer.unk_token_id:
    SEP = 0
print("Tokenizer vocab:", len(tokenizer), "| sentence-end marker id:", SEP)

def encode_doc(doc):
    """stream = sent1 <SEP> sent2 <SEP> ... ; each sentence = (start, end_pos_of_SEP)"""
    toks = tokenizer(doc["sents"], add_special_tokens=False)["input_ids"]
    ids, spans = [], []
    for t in toks:
        t = t[:MAX_SENT_TOK]
        start = len(ids)
        ids += t + [SEP]
        spans.append((start, len(ids) - 1))
    return {"ids": ids, "spans": spans, "sent_toks": [t[:MAX_SENT_TOK] + [SEP] for t in toks]}

for ds in splits.values():
    for d in ds:
        d.update(encode_doc(d))
        if d["labels"] is not None:
            d["y"] = [LAB2ID[l] for l in d["labels"]]
lens = [len(d["ids"]) for d in splits["train"]]
print(f"doc length in tokens: median {int(np.median(lens))}, max {max(lens)}")

def segment_doc(d, max_tok):
    """cut a labelled document into consecutive-sentence segments of <= max_tok tokens (training only)"""
    segs, cur = [], []
    def flush():
        if not cur: return
        st = [d["sent_toks"][i] for i in cur]
        ids, spans = [], []
        for t in st:
            spans.append((len(ids), len(ids) + len(t) - 1)); ids += t
        segs.append({"id": d["id"], "domain": d["domain"], "ids": ids, "spans": spans, "sent_toks": st,
                     "y": [d["y"][i] for i in cur]})
    n_tok = 0
    for i, t in enumerate(d["sent_toks"]):
        if cur and n_tok + len(t) > max_tok:
            flush(); cur, n_tok = [], 0
        cur.append(i); n_tok += len(t)
    flush()
    return segs

train_segments = [seg for d in splits["train"] for seg in segment_doc(d, TRAIN_SEG_TOK)]
print(f"training segments (<= {TRAIN_SEG_TOK} tokens each): {len(train_segments)}")

# %% 5. MACRO-F1 (all labels in label_vocab)
def macro_scores(y_true, y_pred):
    tp, fp, fn = Counter(), Counter(), Counter()
    for t, p in zip(y_true, y_pred):
        if t == p: tp[t] += 1
        else: fp[p] += 1; fn[t] += 1
    per = {}
    for i, l in enumerate(LABELS):
        pr = tp[i] / (tp[i] + fp[i]) if tp[i] + fp[i] else 0.0
        rc = tp[i] / (tp[i] + fn[i]) if tp[i] + fn[i] else 0.0
        per[l] = (pr, rc, 2 * pr * rc / (pr + rc) if pr + rc else 0.0)
    m = lambda k: float(np.mean([v[k] for v in per.values()]))
    return {"precision": m(0), "recall": m(1), "f1": m(2), "per_label_f1": {l: round(v[2], 4) for l, v in per.items()}}

# %% 6. PHASE-1 MODEL DEFINITIONS (unchanged from Phase 1)
def build_model1(cfg):
    # ---------- Model 1: GPT-style ----------
    class CausalSelfAttention(nn.Module):
        def __init__(self, d_model, n_head, dropout):
            super().__init__()
            assert d_model % n_head == 0
            self.d_model = d_model
            self.n_head = n_head
            self.head_dim = d_model // n_head
            self.c_attn = nn.Linear(d_model, 3 * d_model)
            self.c_proj = nn.Linear(d_model, d_model)
            self.attn_dropout = dropout
            self.resid_dropout = nn.Dropout(dropout)

        def forward(self, x):
            B, T, C = x.shape
            qkv = self.c_attn(x)
            q, k, v = qkv.split(self.d_model, dim=2)
            q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
            k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
            v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
            y = F.scaled_dot_product_attention(
                q, k, v, attn_mask=None,
                dropout_p=(self.attn_dropout if self.training else 0.0),
                is_causal=True)
            y = y.transpose(1, 2).contiguous().view(B, T, C)
            y = self.c_proj(y)
            y = self.resid_dropout(y)
            return y


    class MLP(nn.Module):
        def __init__(self, d_model, dropout):
            super().__init__()
            self.fc = nn.Linear(d_model, 4 * d_model)
            self.proj = nn.Linear(4 * d_model, d_model)
            self.dropout = nn.Dropout(dropout)

        def forward(self, x):
            x = self.fc(x)
            x = F.gelu(x)
            x = self.proj(x)
            x = self.dropout(x)
            return x


    class TransformerBlock(nn.Module):
        def __init__(self, d_model, n_head, dropout):
            super().__init__()
            self.ln_1 = nn.LayerNorm(d_model)
            self.attn = CausalSelfAttention(d_model, n_head, dropout)
            self.ln_2 = nn.LayerNorm(d_model)
            self.mlp = MLP(d_model, dropout)

        def forward(self, x):
            x = x + self.attn(self.ln_1(x))
            x = x + self.mlp(self.ln_2(x))
            return x


    class LegalGPT(nn.Module):
        def __init__(self, vocab_size, block_size, n_layer, n_head, d_model, dropout):
            super().__init__()
            self.block_size = block_size
            self.wte = nn.Embedding(vocab_size, d_model)
            self.wpe = nn.Embedding(block_size, d_model)
            self.blocks = nn.ModuleList([TransformerBlock(d_model, n_head, dropout) for _ in range(n_layer)])
            self.ln_f = nn.LayerNorm(d_model)
            self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
            self.lm_head.weight = self.wte.weight
            self.apply(self._init_weights)

        def _init_weights(self, module):
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

        def forward(self, idx, targets=None):
            B, T = idx.shape
            assert T <= self.block_size
            positions = torch.arange(0, T, device=idx.device)
            x = self.wte(idx) + self.wpe(positions)
            for block in self.blocks:
                x = block(x)
            x = self.ln_f(x)
            logits = self.lm_head(x)
            loss = None
            if targets is not None:
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            return logits, loss
    return LegalGPT(**cfg)


def build_model2(cfg):
    # ---------- Model 2: LLaMA-style ----------
    class RMSNorm(nn.Module):
        """x / sqrt(mean(x^2) + eps) * weight  (no mean subtraction, unlike LayerNorm)"""
        def __init__(self, dim, eps=1e-6):
            super().__init__()
            self.weight = nn.Parameter(torch.ones(dim))
            self.eps = eps

        def forward(self, x):
            rms = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
            return (x * rms) * self.weight


    class RotaryEmbedding(nn.Module):
        """RoPE: rotate q and k by position-dependent angles (relative position info)."""
        def __init__(self, dim, max_seq_len, theta=10000.0):
            super().__init__()
            self.dim = dim
            self.max_seq_len = max_seq_len
            inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
            self.register_buffer("inv_freq", inv_freq, persistent=False)
            self._build_cache(max_seq_len)

        def _build_cache(self, seq_len):
            positions = torch.arange(seq_len, device=self.inv_freq.device, dtype=self.inv_freq.dtype)
            freqs = torch.outer(positions, self.inv_freq)
            emb = torch.cat([freqs, freqs], dim=-1)
            self.register_buffer("cos_cached", emb.cos()[None, :, :], persistent=False)
            self.register_buffer("sin_cached", emb.sin()[None, :, :], persistent=False)

        def forward(self, q, k):
            seq_len = q.size(-2)
            if seq_len > self.cos_cached.size(1):
                self._build_cache(seq_len)
            cos = self.cos_cached[:, :seq_len, :].to(q.dtype)
            sin = self.sin_cached[:, :seq_len, :].to(q.dtype)
            return apply_rotary(q, cos, sin), apply_rotary(k, cos, sin)


    def rotate_half(x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2:]
        return torch.cat((-x2, x1), dim=-1)


    def apply_rotary(x, cos, sin):
        return x * cos + rotate_half(x) * sin


    class SwiGLU(nn.Module):
        """down( SiLU(gate(x)) * up(x) ), no bias"""
        def __init__(self, d_model, ffn_dim):
            super().__init__()
            self.gate_proj = nn.Linear(d_model, ffn_dim, bias=False)
            self.up_proj = nn.Linear(d_model, ffn_dim, bias=False)
            self.down_proj = nn.Linear(ffn_dim, d_model, bias=False)

        def forward(self, x):
            return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


    class CausalSelfAttention(nn.Module):
        def __init__(self, d_model, n_heads, max_seq_len):
            super().__init__()
            assert d_model % n_heads == 0
            self.d_model = d_model
            self.n_heads = n_heads
            self.head_dim = d_model // n_heads
            self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
            self.out_proj = nn.Linear(d_model, d_model, bias=False)
            self.rope = RotaryEmbedding(self.head_dim, max_seq_len)

        def forward(self, x):
            B, T, C = x.shape
            q, k, v = self.qkv(x).chunk(3, dim=-1)
            q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
            k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
            v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
            q, k = self.rope(q, k)
            y = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0, is_causal=True)
            y = y.transpose(1, 2).contiguous().view(B, T, C)
            return self.out_proj(y)


    class LLaMABlock(nn.Module):
        def __init__(self, d_model, n_heads, ffn_dim, max_seq_len):
            super().__init__()
            self.attn_norm = RMSNorm(d_model)
            self.attention = CausalSelfAttention(d_model, n_heads, max_seq_len)
            self.ffn_norm = RMSNorm(d_model)
            self.ffn = SwiGLU(d_model, ffn_dim)

        def forward(self, x):
            x = x + self.attention(self.attn_norm(x))   # pre-norm
            x = x + self.ffn(self.ffn_norm(x))
            return x


    class LLaMALanguageModel(nn.Module):
        def __init__(self, cfg):
            super().__init__()
            self.cfg = cfg
            self.token_embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)
            self.blocks = nn.ModuleList([LLaMABlock(cfg.d_model, cfg.n_heads, cfg.ffn_dim, cfg.max_seq_len)
                                         for _ in range(cfg.n_layers)])
            self.final_norm = RMSNorm(cfg.d_model)
            self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
            self.lm_head.weight = self.token_embedding.weight      # weight tying
            self.apply(self._init_weights)

        def _init_weights(self, module):
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

        def forward(self, idx, targets=None):
            x = self.token_embedding(idx)          # no position embedding: RoPE handles position
            for block in self.blocks:
                x = block(x)
            x = self.final_norm(x)
            logits = self.lm_head(x)
            loss = None
            if targets is not None:
                loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
            return logits, loss
        # (your generate() method is not needed: the pipeline's batched generate_batch() is used)
    return LLaMALanguageModel(SimpleNamespace(**cfg))


def build_model3(cfg):
    # ---------- Model 3: LLaMA + GQA + sliding window ----------
    class RMSNorm(nn.Module):
        """x / sqrt(mean(x^2) + eps) * weight  (no mean subtraction, unlike LayerNorm)"""
        def __init__(self, dim, eps=1e-6):
            super().__init__()
            self.weight = nn.Parameter(torch.ones(dim))
            self.eps = eps

        def forward(self, x):
            rms = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
            return (x * rms) * self.weight


    class RotaryEmbedding(nn.Module):
        """RoPE: rotate q and k by position-dependent angles (relative position info)."""
        def __init__(self, dim, max_seq_len, theta=10000.0):
            super().__init__()
            self.dim = dim
            self.max_seq_len = max_seq_len
            inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
            self.register_buffer("inv_freq", inv_freq, persistent=False)
            self._build_cache(max_seq_len)

        def _build_cache(self, seq_len):
            positions = torch.arange(seq_len, device=self.inv_freq.device, dtype=self.inv_freq.dtype)
            freqs = torch.outer(positions, self.inv_freq)
            emb = torch.cat([freqs, freqs], dim=-1)
            self.register_buffer("cos_cached", emb.cos()[None, :, :], persistent=False)
            self.register_buffer("sin_cached", emb.sin()[None, :, :], persistent=False)

        def forward(self, q, k):
            seq_len = q.size(-2)
            if seq_len > self.cos_cached.size(1):
                self._build_cache(seq_len)
            cos = self.cos_cached[:, :seq_len, :].to(q.dtype)
            sin = self.sin_cached[:, :seq_len, :].to(q.dtype)
            return apply_rotary(q, cos, sin), apply_rotary(k, cos, sin)


    def rotate_half(x):
        half = x.shape[-1] // 2
        x1, x2 = x[..., :half], x[..., half:]
        return torch.cat([-x2, x1], dim=-1)


    def apply_rotary(x, cos, sin):
        return x * cos + rotate_half(x) * sin


    class SwiGLU(nn.Module):
        """down( SiLU(gate(x)) * up(x) ), no bias"""
        def __init__(self, d_model, ffn_dim):
            super().__init__()
            self.gate_proj = nn.Linear(d_model, ffn_dim, bias=False)
            self.up_proj = nn.Linear(d_model, ffn_dim, bias=False)
            self.down_proj = nn.Linear(ffn_dim, d_model, bias=False)

        def forward(self, x):
            return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


    class GQASlidingWindowAttention(nn.Module):
        """
        GQA: n_heads query heads share n_kv_heads key/value heads (8 Q -> 2 KV: each KV head serves 4 Q heads).
        Sliding window: token i attends only to tokens j with 0 <= i - j < window_size (causal + local).
        """
        def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, window_size):
            super().__init__()
            assert d_model % n_heads == 0
            assert n_heads % n_kv_heads == 0
            self.d_model = d_model
            self.n_heads = n_heads
            self.n_kv_heads = n_kv_heads
            self.head_dim = d_model // n_heads
            self.window_size = window_size
            self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
            self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
            self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
            self.out_proj = nn.Linear(d_model, d_model, bias=False)
            self.rope = RotaryEmbedding(self.head_dim, max_seq_len)
            # mask[q, k] = True -> q may attend to k   (k <= q  AND  q - k < window)
            positions = torch.arange(max_seq_len)
            distance = positions[:, None] - positions[None, :]
            self.register_buffer("window_mask", (distance >= 0) & (distance < window_size), persistent=False)

        def forward(self, x):
            B, T, C = x.shape
            q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)     # [B, 8, T, D]
            k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)  # [B, 2, T, D]
            v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)  # [B, 2, T, D]
            q, k = self.rope(q, k)
            repeat_factor = self.n_heads // self.n_kv_heads
            k = k.repeat_interleave(repeat_factor, dim=1)                                  # [B, 8, T, D]
            v = v.repeat_interleave(repeat_factor, dim=1)
            mask = self.window_mask[:T, :T]
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=0.0)
            y = y.transpose(1, 2).contiguous().view(B, T, C)
            return self.out_proj(y)


    class LLaMABlock(nn.Module):
        def __init__(self, d_model, n_heads, n_kv_heads, ffn_dim, max_seq_len, window_size):
            super().__init__()
            self.attn_norm = RMSNorm(d_model)
            self.attention = GQASlidingWindowAttention(d_model=d_model, n_heads=n_heads, n_kv_heads=n_kv_heads,
                                                       max_seq_len=max_seq_len, window_size=window_size)
            self.ffn_norm = RMSNorm(d_model)
            self.ffn = SwiGLU(d_model, ffn_dim)

        def forward(self, x):
            x = x + self.attention(self.attn_norm(x))   # pre-norm
            x = x + self.ffn(self.ffn_norm(x))
            return x


    class LLaMALanguageModel(nn.Module):
        def __init__(self, cfg):
            super().__init__()
            self.cfg = cfg
            self.token_embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)
            self.blocks = nn.ModuleList([
                LLaMABlock(d_model=cfg.d_model, n_heads=cfg.n_heads, n_kv_heads=cfg.n_kv_heads, ffn_dim=cfg.ffn_dim,
                           max_seq_len=cfg.max_seq_len, window_size=cfg.window_size)
                for _ in range(cfg.n_layers)])
            self.final_norm = RMSNorm(cfg.d_model)
            self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
            self.lm_head.weight = self.token_embedding.weight      # weight tying
            self.apply(self._init_weights)

        def _init_weights(self, module):
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

        def forward(self, idx, targets=None):
            x = self.token_embedding(idx)          # no position embedding: RoPE handles position
            for block in self.blocks:
                x = block(x)
            x = self.final_norm(x)
            logits = self.lm_head(x)
            loss = None
            if targets is not None:
                loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
            return logits, loss
    return LLaMALanguageModel(SimpleNamespace(**cfg))


def build_model4(cfg):
    # ---------- Model 4: hybrid attention + gated conv ----------
    def rotate_half(x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2:]
        return torch.cat((-x2, x1), dim=-1)


    class RotaryEmbedding(nn.Module):
        """RoPE: rotate q and k by position-dependent angles."""
        def __init__(self, dim, max_seq_len=2048, base=10000):
            super().__init__()
            self.dim = dim
            self.max_seq_len = max_seq_len
            self.base = base
            inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
            self.register_buffer("inv_freq", inv_freq, persistent=False)
            self._build_cache(max_seq_len)

        def _build_cache(self, seq_len):
            positions = torch.arange(seq_len, dtype=self.inv_freq.dtype, device=self.inv_freq.device)
            freqs = torch.outer(positions, self.inv_freq)
            emb = torch.cat([freqs, freqs], dim=-1)
            self.register_buffer("cos_cached", emb.cos()[None, None, :, :], persistent=False)
            self.register_buffer("sin_cached", emb.sin()[None, None, :, :], persistent=False)

        def forward(self, q, k):
            seq_len = q.shape[-2]
            if seq_len > self.cos_cached.shape[-2]:
                self._build_cache(seq_len)
            # FIX: cast to q's dtype. Otherwise fp16 q * fp32 cos -> fp32 q/k while v stays fp16,
            # and scaled_dot_product_attention can fail with a dtype mismatch.
            cos = self.cos_cached[:, :, :seq_len, :].to(q.dtype)
            sin = self.sin_cached[:, :, :seq_len, :].to(q.dtype)
            q = q * cos + rotate_half(q) * sin
            k = k * cos + rotate_half(k) * sin
            return q, k


    class RMSNorm(nn.Module):
        def __init__(self, dim, eps=1e-6):
            super().__init__()
            self.weight = nn.Parameter(torch.ones(dim))
            self.eps = eps

        def forward(self, x):
            rms = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
            return x * rms * self.weight


    class SwiGLU(nn.Module):
        def __init__(self, d_model, hidden_dim):
            super().__init__()
            self.w1 = nn.Linear(d_model, hidden_dim, bias=False)
            self.w2 = nn.Linear(hidden_dim, d_model, bias=False)
            self.w3 = nn.Linear(d_model, hidden_dim, bias=False)

        def forward(self, x):
            return self.w2(F.silu(self.w1(x)) * self.w3(x))


    class CausalSelfAttention(nn.Module):
        """GLOBAL mixer: token i attends to tokens 0..i."""
        def __init__(self, d_model, n_heads, max_seq_len):
            super().__init__()
            assert d_model % n_heads == 0
            self.d_model = d_model
            self.n_heads = n_heads
            self.head_dim = d_model // n_heads
            self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
            self.out_proj = nn.Linear(d_model, d_model, bias=False)
            self.rope = RotaryEmbedding(self.head_dim, max_seq_len=max_seq_len)

        def forward(self, x):
            B, T, C = x.shape
            q, k, v = self.qkv(x).chunk(3, dim=-1)
            q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
            k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
            v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
            q, k = self.rope(q, k)
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=0.0, is_causal=True)
            y = y.transpose(1, 2).contiguous().view(B, T, C)
            return self.out_proj(y)


    class GatedCausalConv(nn.Module):
        """
        LOCAL mixer: Linear(D->2D) -> (value, gate);
        value -> causal depthwise Conv1d (left padding only: position t sees t-k+1 .. t, never t+1);
        out = Linear( conv(value) * sigmoid(gate) ).
        """
        def __init__(self, d_model, kernel_size=5):
            super().__init__()
            self.d_model = d_model
            self.kernel_size = kernel_size
            self.in_proj = nn.Linear(d_model, 2 * d_model, bias=False)
            self.conv = nn.Conv1d(d_model, d_model, kernel_size=kernel_size, groups=d_model, bias=False)
            self.out_proj = nn.Linear(d_model, d_model, bias=False)

        def forward(self, x):
            value, gate = self.in_proj(x).chunk(2, dim=-1)
            value = value.transpose(1, 2)                          # [B, D, T] for Conv1d
            value = F.pad(value, (self.kernel_size - 1, 0))        # LEFT padding only -> causal
            value = self.conv(value).transpose(1, 2)               # back to [B, T, D]
            return self.out_proj(value * torch.sigmoid(gate))


    class HybridBlock(nn.Module):
        def __init__(self, d_model, n_heads, ffn_dim, max_seq_len, conv_kernel, use_attention):
            super().__init__()
            self.use_attention = use_attention
            self.norm1 = RMSNorm(d_model)
            self.mixer = (CausalSelfAttention(d_model=d_model, n_heads=n_heads, max_seq_len=max_seq_len)
                          if use_attention else GatedCausalConv(d_model=d_model, kernel_size=conv_kernel))
            self.norm2 = RMSNorm(d_model)
            self.ffn = SwiGLU(d_model=d_model, hidden_dim=ffn_dim)

        def forward(self, x):
            x = x + self.mixer(self.norm1(x))
            x = x + self.ffn(self.norm2(x))
            return x


    class HybridLM(nn.Module):
        def __init__(self, vocab_size, d_model, n_layers, n_heads, ffn_dim, max_seq_len, conv_kernel):
            super().__init__()
            self.vocab_size = vocab_size
            self.d_model = d_model
            self.token_embedding = nn.Embedding(vocab_size, d_model)
            # even layers -> attention, odd layers -> gated conv
            self.blocks = nn.ModuleList([
                HybridBlock(d_model=d_model, n_heads=n_heads, ffn_dim=ffn_dim, max_seq_len=max_seq_len,
                            conv_kernel=conv_kernel, use_attention=(i % 2 == 0))
                for i in range(n_layers)])
            self.final_norm = RMSNorm(d_model)
            self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
            self.lm_head.weight = self.token_embedding.weight      # weight tying
            self.apply(self._init_weights)

        def _init_weights(self, module):
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

        def forward(self, input_ids, targets=None):
            x = self.token_embedding(input_ids)
            for block in self.blocks:
                x = block(x)
            x = self.final_norm(x)
            logits = self.lm_head(x)
            loss = None
            if targets is not None:
                # FIX: the pipeline's targets are ALREADY shifted by one (y = x[1:]).
                # Your version shifted again (logits[:, :-1] vs targets[:, 1:]) -> the model
                # would learn to predict 2 tokens ahead. Same loss as Models 1-3 now.
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            return logits, loss
    return HybridLM(**cfg)


BUILDERS = {"gpt": build_model1, "llama": build_model2, "gqa": build_model3, "hybrid": build_model4}

# %% 7. PEFT MODULES (all from scratch)
class LoRALinear(nn.Module):
    """LoRA:   y = W x + s * B A x,  s = alpha/r.   rsLoRA: same but s = alpha/sqrt(r) (rank-stabilised)."""
    def __init__(self, base, r, alpha, rank_stabilized=False):
        super().__init__()
        self.base = base
        self.scale = alpha / math.sqrt(r) if rank_stabilized else alpha / r
        self.A = nn.Parameter(torch.randn(r, base.in_features) / math.sqrt(base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(x, self.A), self.B)


class LoRMALinear(nn.Module):
    """LoRMA (multiplicative):  y = (I + s * B A) W x.  Check the exact variant against the LoRMA paper."""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.A = nn.Parameter(torch.randn(r, base.out_features) / math.sqrt(base.out_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x):
        y = self.base(x)
        return y + self.scale * F.linear(F.linear(y, self.A), self.B)


class Quant4Linear(nn.Module):
    """Frozen weight stored in 4 bits (QLoRA-style): blockwise absmax, 2 codes packed per byte.
    (QLoRA uses the NF4 code book; this uses uniform 4-bit levels -7..7 - say so in the report.)"""
    def __init__(self, base, block=64):
        super().__init__()
        W = base.weight.data.float()
        self.out_features, self.in_features = W.shape
        flat = W.flatten()
        pad = (-flat.numel()) % block
        flat = torch.cat([flat, flat.new_zeros(pad)]).view(-1, block)
        scale = flat.abs().amax(1, keepdim=True).clamp(min=1e-8) / 7.0
        q = torch.clamp(torch.round(flat / scale), -7, 7).to(torch.int16) + 8      # 1..15
        q = q.flatten().to(torch.uint8)
        if q.numel() % 2: q = torch.cat([q, q.new_zeros(1)])
        self.register_buffer("packed", (q[0::2] | (q[1::2] << 4)).contiguous())
        self.register_buffer("scale", scale.half())
        self.numel, self.block = W.numel(), block
        self.bias = None if base.bias is None else nn.Parameter(base.bias.data.clone(), requires_grad=False)

    def weight(self):
        q = torch.stack([self.packed & 15, self.packed >> 4], 1).flatten()[: self.scale.numel() * self.block]
        W = (q.view(-1, self.block).float() - 8.0) * self.scale.float()
        return W.flatten()[: self.numel].view(self.out_features, self.in_features)

    def forward(self, x):
        return F.linear(x, self.weight().to(x.dtype), None if self.bias is None else self.bias.to(x.dtype))


class AdaLoRALinear(nn.Module):
    """AdaLoRA:  y = W x + s * P diag(lambda * mask) Q x  (SVD-form update).
    Ranks are pruned during training by importance |lambda * dL/dlambda| (EMA), down to a global budget."""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.P = nn.Parameter(torch.randn(base.out_features, r) * 0.02)
        self.Q = nn.Parameter(torch.randn(r, base.in_features) * 0.02)
        self.lam = nn.Parameter(torch.zeros(r))
        self.register_buffer("mask", torch.ones(r))
        self.register_buffer("importance", torch.zeros(r))

    def forward(self, x):
        h = F.linear(x, self.Q) * (self.lam * self.mask)
        return self.base(x) + self.scale * F.linear(h, self.P)

    def orth_reg(self):
        r = self.lam.numel()
        I = torch.eye(r, device=self.P.device)
        return ((self.P.T @ self.P - I) ** 2).sum() + ((self.Q @ self.Q.T - I) ** 2).sum()


class IA3Linear(nn.Module):
    """IA3: learned rescaling vector (init 1). mode="out": scales the layer output (keys/values);
    mode="in": scales the input of the FFN down-projection (= the FFN hidden activations)."""
    def __init__(self, base, mode):
        super().__init__()
        self.base, self.mode = base, mode
        self.l = nn.Parameter(torch.ones(base.out_features if mode == "out" else base.in_features))

    def forward(self, x):
        return self.base(x) * self.l if self.mode == "out" else self.base(x * self.l)


def wrap_linears(module, fn):
    """replace every nn.Linear `child` (attribute name `name`) by fn(child, name) unless it returns None"""
    for name, child in module.named_children():
        if isinstance(child, nn.Linear):
            new = fn(child, name)
            if new is not None:
                setattr(module, name, new)
        else:
            wrap_linears(child, fn)

IA3_OUT = {"c_attn", "qkv", "k_proj", "v_proj"}   # attention K/V projections (fused qkv: whole output)
IA3_IN = {"proj", "down_proj", "w2"}               # FFN down-projections (their input = FFN activations)

# %% 8. BACKBONE FORWARD WITH OPTIONAL SOFT PROMPT / PREFIX + SENTENCE CLASSIFIER
def emb_module(bb):
    return bb.wte if hasattr(bb, "wte") else bb.token_embedding

def final_norm(bb):
    return bb.ln_f if hasattr(bb, "ln_f") else bb.final_norm


class RRModel(nn.Module):
    def __init__(self, backbone, d_model, n_labels, method, head, n_prompt):
        super().__init__()
        self.bb, self.method, self.head_type = backbone, method, head
        self.n_prompt = n_prompt if method in ("prompt", "prefix") else 0
        if method == "prompt":     # soft prompt: P trainable vectors in front of the input embeddings
            self.prompt = nn.Parameter(torch.randn(n_prompt, d_model) * 0.02)
        if method == "prefix":     # prefix (deep prompt, P-tuning-v2 style): trainable vectors at EVERY layer
            self.prefix = nn.Parameter(torch.randn(len(backbone.blocks) + 1, n_prompt, d_model) * 0.02)
        if head == "bigru":
            self.gru = nn.GRU(d_model, d_model // 2, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(0.1)
        self.cls = nn.Linear(d_model, n_labels)

    def hidden(self, ids):
        """ids (B, T) -> hidden states (B, T, d) for the real tokens"""
        bb, P = self.bb, self.n_prompt
        x = emb_module(bb)(ids)
        if P:
            pre = self.prompt if self.method == "prompt" else self.prefix[0]
            x = torch.cat([pre.unsqueeze(0).expand(ids.size(0), -1, -1).to(x.dtype), x], 1)
        if hasattr(bb, "wpe"):
            x = x + bb.wpe(torch.arange(x.size(1), device=ids.device))
        for i, blk in enumerate(bb.blocks):
            if self.method == "prefix" and i > 0:
                x = torch.cat([self.prefix[i].unsqueeze(0).expand(ids.size(0), -1, -1).to(x.dtype), x[:, P:]], 1)
            x = blk(x)
        return final_norm(bb)(x)[:, P:]

    def classify(self, sent_vecs):
        """(S, d) sentence vectors of ONE document -> (S, n_labels) logits"""
        h = sent_vecs.unsqueeze(0)
        if self.head_type == "bigru":
            h, _ = self.gru(h.float())
        return self.cls(self.drop(h))[0]


def build_rr_model(model_type, p1_cfg, p1_state, method, head, n_prompt, keep_every=None):
    bb = BUILDERS[model_type](p1_cfg)
    if p1_state is not None:
        bb.load_state_dict(p1_state)
    bb.lm_head = nn.Identity()
    if keep_every:
        bb.blocks = nn.ModuleList(list(bb.blocks)[::keep_every])
    d = emb_module(bb).weight.shape[1]
    m = RRModel(bb, d, len(LABELS), method, head, n_prompt)
    if method != "full":
        for p in bb.parameters(): p.requires_grad = False
    blocks = bb.blocks
    if method in ("lora", "rslora"):
        wrap_linears(blocks, lambda lin, n: LoRALinear(lin, LORA_R, LORA_ALPHA, rank_stabilized=(method == "rslora")))
    elif method == "lorma":
        wrap_linears(blocks, lambda lin, n: LoRMALinear(lin, LORA_R, LORA_ALPHA))
    elif method == "qlora":
        wrap_linears(blocks, lambda lin, n: LoRALinear(Quant4Linear(lin, QUANT_BLOCK), LORA_R, LORA_ALPHA))
    elif method == "adalora":
        wrap_linears(blocks, lambda lin, n: AdaLoRALinear(lin, ADA_R_INIT, LORA_ALPHA))
    elif method == "ia3":
        wrap_linears(blocks, lambda lin, n: IA3Linear(lin, "out") if n in IA3_OUT
                     else IA3Linear(lin, "in") if n in IA3_IN else None)
    elif method not in ("full", "prompt", "prefix"):
        raise ValueError(method)
    return m

# %% 9. LOAD PHASE-1 WEIGHTS
def as_ckpt_file(path):
    """Kaggle sometimes auto-extracts an uploaded .pt (it is a zip inside) into a FOLDER.
    If `path` is such a folder, re-pack it into a normal .pt file that torch.load can read."""
    if not os.path.isdir(path):
        return path
    root = next((r for r, _, f in os.walk(path) if "data.pkl" in f), None)
    assert root, f"{path} is a folder without data.pkl - not an extracted checkpoint"
    out = os.path.join("/kaggle/working", os.path.basename(path.rstrip("/")) + "_repacked.pt")
    with zipfile.ZipFile(out, "w", zipfile.ZIP_STORED) as z:
        for r, _, files in os.walk(root):
            for f in files:
                full = os.path.join(r, f)
                z.write(full, "archive/" + os.path.relpath(full, root))
    print(f"Re-packed extracted checkpoint folder {path} -> {out}")
    return out
ck = torch.load(as_ckpt_file(PHASE1_CKPT), map_location="cpu")
P1_CFG = ck["config"]
print("Phase-1 config:", P1_CFG)
assert P1_CFG["vocab_size"] == len(tokenizer), "Tokenizer doesn't match this Phase-1 model"
CTX = P1_CFG.get("block_size", P1_CFG.get("max_seq_len"))
assert MAX_LEN + (N_PROMPT if FT_METHOD in ("prompt", "prefix") else 0) <= CTX, "MAX_LEN (+prompt) > Phase-1 context"

RUN_CFG = dict(model_type=MODEL_TYPE, ft_method=FT_METHOD, head=HEAD, pool=POOL, context=CONTEXT,
               n_prompt=N_PROMPT, lora_r=LORA_R, lora_alpha=LORA_ALPHA, ada_r_init=ADA_R_INIT,
               keep_every=KD_KEEP_EVERY if KD_TEACHER else None, phase1_cfg=P1_CFG)

teacher = None
if KD_TEACHER:
    t = torch.load(as_ckpt_file(KD_TEACHER), map_location="cpu")
    tc = t["run_cfg"]
    assert tc["context"] == CONTEXT, "teacher and student must use the same CONTEXT mode"
    for k in ("lora_r", "lora_alpha", "ada_r_init", "n_prompt"):
        globals()[k.upper()] = tc[k]               # rebuild the teacher with ITS adapter sizes
    teacher = build_rr_model(tc["model_type"], tc["phase1_cfg"], None, tc["ft_method"], tc["head"],
                             tc["n_prompt"], tc.get("keep_every"))
    teacher.load_state_dict(t["state"])
    teacher = teacher.to(DEVICE).eval()
    for p in teacher.parameters(): p.requires_grad = False
    T_POOL = tc["pool"]
    n_teacher = sum(p.numel() for p in teacher.parameters()) + sum(q.numel for q in teacher.modules() if isinstance(q, Quant4Linear))
    print(f"KD teacher: {tc['model_type']} + {tc['ft_method']} ({n_teacher:,} params)")
    FT_METHOD = "full"                              # the student is fully trained (smaller body)
    RUN_CFG["ft_method"] = "full"

model = build_rr_model(MODEL_TYPE, P1_CFG, ck["model"], FT_METHOD, HEAD, N_PROMPT,
                       KD_KEEP_EVERY if KD_TEACHER else None).to(DEVICE)
del ck
def count_params(m):
    """parameters + 4-bit-quantised frozen weights (QLoRA stores those as buffers)"""
    return sum(p.numel() for p in m.parameters()) + sum(q.numel for q in m.modules() if isinstance(q, Quant4Linear))
def weight_mb(m):
    return (sum(p.numel() * p.element_size() for p in m.parameters())
            + sum(q.packed.numel() + q.scale.numel() * 2 for q in m.modules() if isinstance(q, Quant4Linear))) / 2**20
n_total = count_params(model)
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
MODEL_MB = weight_mb(model)
print(f"{FT_METHOD}{' (KD student)' if KD_TEACHER else ''}: trainable {n_train:,} / {n_total:,} "
      f"({100 * n_train / n_total:.2f}%) | weights {MODEL_MB:.1f} MB | head={HEAD} pool={POOL} context={CONTEXT}")
if KD_TEACHER:
    print(f"KD: student {n_total:,} params = {100 * n_total / n_teacher:.0f}% of teacher")

# %% 10. DOCUMENT -> SENTENCE LOGITS
USE_AMP = DEVICE.type == "cuda"
AMP_DTYPE = torch.bfloat16 if USE_AMP and torch.cuda.is_bf16_supported() else torch.float16
autocast = lambda: torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP)
STEP = MAX_LEN - OVERLAP

def pad_batch(seqs):
    T = max(len(s) for s in seqs)
    x = torch.zeros(len(seqs), T, dtype=torch.long)
    for i, s in enumerate(seqs):
        x[i, :len(s)] = torch.tensor(s)   # right padding: safe for causal models
    return x.to(DEVICE)

def sentence_vectors(m, doc, pool):
    """(S, d) one vector per sentence"""
    if CONTEXT == "sentence":
        vecs = []
        st = doc["sent_toks"]
        for s in range(0, len(st), WIN_BATCH * 4):
            chunk = st[s:s + WIN_BATCH * 4]
            with autocast():
                h = m.hidden(pad_batch(chunk))
            for i, toks in enumerate(chunk):
                vecs.append(h[i, len(toks) - 1] if pool == "last" else h[i, :len(toks)].mean(0))
        return torch.stack(vecs).float()
    # doc mode: windows over the sentence stream; each sentence is read in the window that
    # contains its end marker with the MOST left context (window "owns" positions >= own_from)
    ids, spans = doc["ids"], doc["spans"]
    wins, start = [], 0
    while True:
        end = min(start + MAX_LEN, len(ids))
        wins.append((start, end, 0 if start == 0 else OVERLAP))
        if end == len(ids): break
        start += STEP
    owner = []
    for s0, e in spans:
        w = next(k for k, (ws, we, own) in enumerate(wins) if ws + own <= e < we)
        owner.append(w)
    H = []
    for b in range(0, len(wins), WIN_BATCH):
        chunk = wins[b:b + WIN_BATCH]
        with autocast():
            H += list(m.hidden(pad_batch([ids[ws:we] for ws, we, _ in chunk])))
    vecs = []
    for (s0, e), w in zip(spans, owner):
        ws = wins[w][0]
        if pool == "last":
            vecs.append(H[w][e - ws])
        else:
            vecs.append(H[w][max(s0, ws) - ws: e - ws + 1].mean(0))
    return torch.stack(vecs).float()

def doc_logits(m, doc, pool=POOL):
    return m.classify(sentence_vectors(m, doc, pool)).float()

# %% 11. TRAINING
if CLASS_WEIGHT == "sqrt_inv":
    freq = np.array([cnt[l] for l in LABELS], dtype=np.float64) + 1.0
    w = 1.0 / np.sqrt(freq); w = w / w.mean()
    class_w = torch.tensor(w, dtype=torch.float32, device=DEVICE)
else:
    class_w = None

new_ids = {id(p) for n, p in model.named_parameters()
           if not n.startswith("bb.") or n.endswith((".A", ".B", ".P", ".Q", ".lam", ".l"))}
groups = [{"params": [p for p in model.parameters() if p.requires_grad and id(p) in new_ids], "lr": LR_NEW},
          {"params": [p for p in model.parameters() if p.requires_grad and id(p) not in new_ids], "lr": LR_BODY}]
optimizer = torch.optim.AdamW([g for g in groups if g["params"]], weight_decay=WEIGHT_DECAY)
train_docs = train_segments
total_steps = EPOCHS * math.ceil(len(train_docs) / DOCS_PER_STEP)
warmup = max(1, int(0.06 * total_steps))
sched = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lambda s: min(1.0, (s + 1) / warmup) * max(0.0, (total_steps - s) / max(1, total_steps - warmup)))
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and AMP_DTYPE == torch.float16)

ada_mods = [m_ for m_ in model.modules() if isinstance(m_, AdaLoRALinear)]
ADA_BUDGET0 = sum(m_.lam.numel() for m_ in ada_mods)
ADA_TARGET = ADA_R_TARGET * len(ada_mods)
ADA_T0, ADA_T1, ADA_EVERY, ADA_BETA = int(0.1 * total_steps), int(0.7 * total_steps), 10, 0.85

def adalora_update(step):
    """EMA importance of every singular value; keep the top-`budget` globally (cubic schedule)."""
    for m_ in ada_mods:
        if m_.lam.grad is not None:
            m_.importance.mul_(ADA_BETA).add_((1 - ADA_BETA) * (m_.lam * m_.lam.grad).abs().detach())
    if step < ADA_T0 or step % ADA_EVERY: return
    prog = min(1.0, (step - ADA_T0) / max(1, ADA_T1 - ADA_T0))
    budget = int(ADA_TARGET + (ADA_BUDGET0 - ADA_TARGET) * (1 - prog) ** 3)
    allimp = torch.cat([m_.importance for m_ in ada_mods])
    if budget < allimp.numel():
        thr = torch.topk(allimp, budget).values[-1]
        for m_ in ada_mods:
            m_.mask.copy_((m_.importance >= thr).float())

@torch.no_grad()
def evaluate(docs):
    model.eval()
    yt, yp, per_dom = [], [], defaultdict(lambda: ([], []))
    for d in docs:
        p = doc_logits(model, d).argmax(-1).tolist()
        yt += d["y"]; yp += p
        per_dom[d["domain"]][0].extend(d["y"]); per_dom[d["domain"]][1].extend(p)
    res = macro_scores(yt, yp)
    res["per_domain_f1"] = {k: round(macro_scores(*v)["f1"], 4) for k, v in per_dom.items()}
    return res

BEST_PATH = os.path.join(OUTPUT_DIR, "best_rr.pt")
best_f1, history, step, t0 = -1.0, [], 0, time.time()
if DEVICE.type == "cuda": torch.cuda.reset_peak_memory_stats()
for epoch in range(1, EPOCHS + 1):
    model.train()
    random.shuffle(train_docs)
    tot, n = 0.0, 0
    optimizer.zero_grad(set_to_none=True)
    for i, d in enumerate(train_docs):
        logits = doc_logits(model, d)
        y = torch.tensor(d["y"], device=DEVICE)
        loss = F.cross_entropy(logits, y, weight=class_w)
        if teacher is not None:
            with torch.no_grad():
                t_logits = doc_logits(teacher, d, T_POOL)
            kl = F.kl_div(F.log_softmax(logits / KD_T, -1), F.softmax(t_logits / KD_T, -1), reduction="batchmean")
            loss = KD_ALPHA * loss + (1 - KD_ALPHA) * KD_T ** 2 * kl
        if ada_mods:
            loss = loss + 0.1 * sum(m_.orth_reg() for m_ in ada_mods) / len(ada_mods)
        scaler.scale(loss / DOCS_PER_STEP).backward()
        tot += loss.item(); n += 1
        if (i + 1) % DOCS_PER_STEP == 0 or i + 1 == len(train_docs):
            scaler.unscale_(optimizer)
            if ada_mods: adalora_update(step)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(optimizer); scaler.update(); sched.step()
            optimizer.zero_grad(set_to_none=True)
            step += 1
    res = evaluate(splits["dev"])
    extra = f" | active AdaLoRA ranks {int(sum(m_.mask.sum().item() for m_ in ada_mods))}/{ADA_BUDGET0}" if ada_mods else ""
    history.append({"epoch": epoch, "train_loss": tot / n, **{k: res[k] for k in ("precision", "recall", "f1", "per_domain_f1")}})
    print(f"epoch {epoch} | loss {tot / n:.4f} | dev macro P {res['precision']:.4f} R {res['recall']:.4f} "
          f"F1 {res['f1']:.4f} | per domain {res['per_domain_f1']} | {time.time() - t0:.0f}s{extra}")
    if res["f1"] > best_f1:
        best_f1, best_res = res["f1"], res
        torch.save({"state": model.state_dict(), "run_cfg": RUN_CFG}, BEST_PATH)
        print("  >> best model saved")

TRAIN_MIN = (time.time() - t0) / 60
PEAK_GB = torch.cuda.max_memory_allocated() / 1024**3 if DEVICE.type == "cuda" else None
print(f"\nBest dev macro-F1 {best_f1:.4f} | per domain {best_res['per_domain_f1']} | "
      f"{TRAIN_MIN:.1f} min | peak GPU {PEAK_GB} GB")
print("Per-label F1:", best_res["per_label_f1"])

# %% 12. TEST PREDICTIONS + prediction.json + ZIP
model.load_state_dict(torch.load(BEST_PATH, map_location=DEVICE)["state"])
model.eval()
predictions = {}
with torch.no_grad():
    for d in splits["test"]:
        p = doc_logits(model, d).argmax(-1).tolist()
        predictions[d["id"]] = [LABELS[i] for i in p]
for d in splits["test"]:
    assert len(predictions[d["id"]]) == len(d["sents"]), f"{d['id']}: wrong number of predictions"
print(f"{len(predictions)} test docs | {sum(map(len, predictions.values()))} sentences | "
      f"predicted label counts: {dict(Counter(l for v in predictions.values() for l in v))}")

pred_path = os.path.join(OUTPUT_DIR, "prediction.json")
with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False)
zip_path = os.path.join(OUTPUT_DIR, "submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(pred_path, arcname="prediction.json")
print("Submission ready:", zip_path)

# %% 13. RUN SUMMARY (experiment log / report)
with open(os.path.join(OUTPUT_DIR, "run_summary.json"), "w") as f:
    json.dump({**{k: v for k, v in RUN_CFG.items() if k != "phase1_cfg"}, "kd_teacher": KD_TEACHER,
               "trainable_params": n_train, "total_params": n_total, "weights_mb": round(MODEL_MB, 1),
               "teacher_params": n_teacher if KD_TEACHER else None,
               "max_len": MAX_LEN, "epochs": EPOCHS, "lr_body": LR_BODY, "lr_new": LR_NEW,
               "class_weight": CLASS_WEIGHT, "dev_best": best_res, "history": history,
               "train_minutes": round(TRAIN_MIN, 1), "peak_gpu_gb": PEAK_GB}, f, indent=2)
print("Done.")

Device: cuda Tesla T4


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

/tmp/cs781_data/Phase_3_RR/Train/CL_dev.json 0.34 MB
/tmp/cs781_data/Phase_3_RR/Train/CL_train.json 2.46 MB
/tmp/cs781_data/Phase_3_RR/Train/IT_dev.json 0.17 MB
/tmp/cs781_data/Phase_3_RR/Train/IT_train.json 1.51 MB
/tmp/cs781_data/Phase_3_RR/Train/label_vocab.json 0.00 MB
/tmp/cs781_data/Phase_3_RR/Test/CL_test.json 0.28 MB
/tmp/cs781_data/Phase_3_RR/Test/IT_test.json 0.19 MB

Train/CL_train.json: list 40 | first: {"id": "CCI_All_India_Distillers_Association_vs_Haldyn_Glass_GCO100028COM855733", "text": ["Consequent upon the repeal of the MRTP Act this complaint has been received by transfer on 04.03.2010 from the office of DGIR, MRTP Commission under Section 66(6) of the Competition Act, 2002.", "The facts as stated in the complaint, in brief, are: 2 .1 This complaint was filed by the All India Distillers A...

Test/CL_test.json: dict 5 | first: ["HC_Raghupati_Singhania__vs__Competition_Commission_ofDE201616051616063761COM513489", ["Manmohan, J. 1 .", "Present writ petition has been f